In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from spark_config import SparkConfig
from logger import LoggerConfig
from io_config import IOConfig
from platform_app import PlatformApp

In [2]:
logger = LoggerConfig().setup_logger(log_dir=IOConfig.LOG_DIR)
spark = SparkConfig.create_spark(app_name="SQL in Spark", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="databricks_toturial")

2026-03-05 10:17:13 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-05 10:17:15 | INFO     | Connected to Databricks via Spark Connect.
2026-03-05 10:17:15 | INFO     | Initializing Data Platform...
2026-03-05 10:17:15 | INFO     | Spark session initialized


In [6]:
df = (
        spark.read.
        format("csv").
        option("header", True).
        option("inferSchema", True).
        csv("/Volumes/databricks_toturial/source/source_data/movies")
    )
df.show(n=10, truncate=False)

+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|title                                      |industry |release_year|imdb_rating|studio                   |budget |revenue |unit     |currency|language|
+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|Pather Panchali                            |Bollywood|1955        |8.3        |Government of West Bengal|70000.0|100000.0|Thousands|INR     |Bengali |
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios           |200.0  |954.8   |Millions |USD     |English |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios           |165.0  |644.8   |Millions |USD     |English |
|Thor: Ragnarok                             |Hollywood|2017        |7.9        |Marvel S

In [7]:
df.createOrReplaceTempView("temp_movies")
df_marvel = spark.sql("""
    SELECT  *
    FROM    temp_movies
    WHERE   studio = 'Marvel Studios'
""")
df_marvel.show(n=10, truncate=False)

+-------------------------------------------+---------+------------+-----------+--------------+------+-------+--------+--------+--------+
|title                                      |industry |release_year|imdb_rating|studio        |budget|revenue|unit    |currency|language|
+-------------------------------------------+---------+------------+-----------+--------------+------+-------+--------+--------+--------+
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios|200.0 |954.8  |Millions|USD     |English |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios|165.0 |644.8  |Millions|USD     |English |
|Thor: Ragnarok                             |Hollywood|2017        |7.9        |Marvel Studios|180.0 |854.0  |Millions|USD     |English |
|Thor: Love and Thunder                     |Hollywood|2022        |6.8        |Marvel Studios|250.0 |670.0  |Millions|USD     |English |
|Avengers: Endgame                

## SQL Using %sql

In [ ]:
%sql
SELECT  *
FROM    temp_movies
WHERE   studio = 'Marvel Studios'

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
0,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7,Marvel Studios,200.0,954.8,Millions,USD,English
1,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.0,644.8,Millions,USD,English
2,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English
3,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.0,670.0,Millions,USD,English
4,Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English
5,Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English
6,Captain America: The First Avenger,Hollywood,2011,6.9,Marvel Studios,216.7,370.6,Millions,USD,English
7,Captain America: The Winter Soldier,Hollywood,2014,7.8,Marvel Studios,177.0,714.4,Millions,USD,English


In [10]:
logger = LoggerConfig().setup_logger(log_dir=IOConfig.LOG_DIR)
spark = SparkConfig.create_spark(app_name="SQL in Spark", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="databricks_toturial")

2026-03-05 10:41:38 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-05 10:41:38 | INFO     | Connected to Databricks via Spark Connect.
2026-03-05 10:41:38 | INFO     | Initializing Data Platform...
2026-03-05 10:41:38 | INFO     | Spark session initialized


In [13]:
from pyspark.sql import functions as F, types as T

data = [
    ("2017-01-01", 32.0, 6.0, "Rain"),
    ("2017-01-04", None, 9.0, "Sunny"),
    ("2017-01-05", 28.0, None, "Snow"),
    ("2017-01-06", None, 7.0, None),
    ("2017-01-07", 32.0, None, "Rain"),
    ("2017-01-08", None, None, "Sunny"),
    ("2017-01-09", None, None, None),
    ("2017-01-10", 34.1, 8.1, "Cloudy"),
    ("2017-01-11", 40.0, 12.0, "Sunny"),
]

schema = "day string, temperature double, windspeed double, event string"

df = spark.createDataFrame(data, schema)
df = df.withColumn("day", F.to_date("day", "yyyy-MM-dd"))  # normalize to DateType

display(df)

,day,temperature,windspeed,event
0,2017-01-01,32.0,6.0,Rain
1,2017-01-04,NaN,9.0,Sunny
2,2017-01-05,28.0,NaN,Snow
3,2017-01-06,NaN,7.0,NaN
4,2017-01-07,32.0,NaN,Rain
5,2017-01-08,NaN,NaN,Sunny
6,2017-01-09,NaN,NaN,NaN
7,2017-01-10,34.1,8.1,Cloudy
8,2017-01-11,40.0,12.0,Sunny


In [16]:
df.createOrReplaceTempView("weather")

In [19]:
%sql
SELECT      event,
            ROUND(AVG(temperature), 1) AS avg_temp
FROM        weather
GROUP BY    event
ORDER BY    avg_temp DESC;

,event,avg_temp
0,Sunny,40.0
1,Cloudy,34.1
2,Rain,32.0
3,Snow,28.0
4,NaN,NaN
